In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║  RBKT Cybersecurity — Orquestador CMMI + Heatmap de Riesgos                  ║
║  1_FULL_ASSESSMENT.py                                                        ║
║                                                                              ║
║  FUENTE DE DATOS: únicamente el CSV (3.PRUEBA.csv)                           ║
║  SALIDA:          CMMI_RBKT_Report.pptx (generado por 2_POWER_POINT.js)      ║
║                                                                              ║
║  Uso:                                                                        ║
║    python 1_FULL_ASSESSMENT.py                        (usa defaults)         ║
║    python 1_FULL_ASSESSMENT.py --csv datos.csv --out informe.pptx            ║
║                                                                              ║
║  Dependencias:                                                               ║
║    pip install pandas numpy                                                  ║
║    npm install pptxgenjs   (para el script Node)                             ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
import argparse
import base64
import json
import math
import subprocess
import numpy as np
import pandas as pd
import re

# ══════════════════════════════════════════════════════════════════════════════
# CARGA DE VARIABLES DE ENTORNO
# ══════════════════════════════════════════════════════════════════════════════
load_dotenv()

OLLAMA_BASE_URL  = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL     = os.getenv("OLLAMA_MODEL",    "llama3.1:8b")
OLLAMA_TEMP      = float(os.getenv("OLLAMA_TEMPERATURE", "0.35"))
OLLAMA_CTX       = int(os.getenv("OLLAMA_NUM_CTX",      "4096"))
DEFAULT_CSV      = os.getenv("CSV_PATH",   "3.PRUEBA.csv")
DEFAULT_OUT      = os.getenv("OUTPUT_PPTX", "CMMI_RBKT_Report.pptx")

# ══════════════════════════════════════════════════════════════════════════════
# SECCIÓN 1: UTILIDADES GENERALES
# ══════════════════════════════════════════════════════════════════════════════

def _b64(path: str) -> str:
    try:
        with open(path, "rb") as f:
            return base64.b64encode(f.read()).decode()
    except FileNotFoundError:
        return ""


# ══════════════════════════════════════════════════════════════════════════════
# SECCIÓN 2: CARGA Y LIMPIEZA DEL CSV
# ══════════════════════════════════════════════════════════════════════════════

def load_and_clean(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df = df[["PROCESO", "SUB-PROCESO", "NIVEL DE MADUREZ",
             "PONDERACION NIVEL DE MADUREZ", "%"]].copy()
    df.columns = ["PROC", "SUB_PROC", "NVL_MADUREZ", "PND_MADUREZ", "PCTG"]
    df["PCTG"] = (
        df["PCTG"].astype(str)
        .str.replace("%", "", regex=False)
        .str.strip()
        .replace("nan", np.nan)
    )
    df["PCTG"] = pd.to_numeric(df["PCTG"], errors="coerce")
    df["NVL_MADUREZ"] = df["NVL_MADUREZ"].astype(str).str.strip()
    df = df.dropna(subset=["PROC", "SUB_PROC", "NVL_MADUREZ"])
    return df.reset_index(drop=True)


# ══════════════════════════════════════════════════════════════════════════════
# SECCIÓN 3: MÓDULO CMMI
# ══════════════════════════════════════════════════════════════════════════════

def _analizar_proceso(df: pd.DataFrame, proceso: str) -> tuple:
    df_proc = df[df["PROC"] == proceso]
    calificacion_sub = df_proc.groupby("SUB_PROC")["PCTG"].mean().reset_index()
    calificacion_sub["PCTG"] = calificacion_sub["PCTG"].round(2)
    promedio = df_proc["PCTG"].mean()
    orden = ["1-Inexistente", "2-Inicial", "3-Definida", "4-Optimizada"]
    madurez = df_proc.groupby(["PROC", "NVL_MADUREZ"]).size().reset_index(name="recuento")
    madurez["NVL_MADUREZ"] = pd.Categorical(
        madurez["NVL_MADUREZ"], categories=orden, ordered=True
    )
    madurez = madurez.sort_values(["PROC", "NVL_MADUREZ"])
    return calificacion_sub, promedio, madurez


def build_report_data(df: pd.DataFrame) -> dict:
    procesos = sorted(df["PROC"].unique())
    orden_madurez = ["1-Inexistente", "2-Inicial", "3-Definida", "4-Optimizada"]
    proc_scores = []
    for proc in procesos:
        subs, avg, madurez = _analizar_proceso(df, proc)
        subs_list = [
            {"subproceso": str(r["SUB_PROC"]), "pctg": float(r["PCTG"])}
            for _, r in subs.iterrows()
        ]
        madurez_dict = {lvl: 0 for lvl in orden_madurez}
        for _, row in madurez.iterrows():
            madurez_dict[str(row["NVL_MADUREZ"])] = int(row["recuento"])
        proc_scores.append({
            "name":           proc,
            "avgPct":         round(float(avg), 2),
            "subs":           subs_list,
            "maturityCounts": madurez_dict,
        })
    proc_scores.sort(key=lambda x: x["avgPct"])
    global_avg = round(float(df["PCTG"].mean()), 2)
    return {"processes": proc_scores, "globalAvg": global_avg}


# ══════════════════════════════════════════════════════════════════════════════
# SECCIÓN 4: MÓDULO DE RIESGOS
# ══════════════════════════════════════════════════════════════════════════════

PONDERACION = {
    "1-Inexistente": 0.00,
    "2-Inicial":     0.33,
    "3-Definida":    0.66,
    "4-Optimizada":  1.00,
}

REGLAS_RIESGOS = {
    1:  [2, 3, 4, 5, 9],
    2:  [3, 4, 9, 30, 32],
    3:  [9, 10, 12, 13, 15, 32],
    4:  [9, 10, 11, 12, 13, 57, 58, 59, 63, 65],
    5:  [6, 8, 9, 20],
    6:  [45, 46, 47, 15, 16, 22],
    7:  [7, 37],
    8:  [35, 36, 62, 63, 64, 66, 67],
    9:  [34, 62, 63, 64],
    10: [15, 16, 17, 21, 33],
    11: [25, 26, 28, 47, 48, 56, 59, 63, 64, 65, 66, 67],
    12: [30, 31, 38, 62, 65],
    13: [18, 19],
    14: [25, 26, 27, 28, 32, 62, 63, 64, 65],
    15: [21, 33],
    16: [7, 9, 23, 24],
    17: [29, 36],
    18: [11, 12, 13, 50, 51, 52, 53, 54, 55, 61],
    19: [2, 3, 4, 5, 9, 10, 11, 12, 13, 14, 44],
}

IMPACTOS_RIESGOS = {
    1: 3,  2: 2,  3: 3,  4: 3,  5: 2,
    6: 2,  7: 3,  8: 3,  9: 3, 10: 2,
   11: 3, 12: 1, 13: 3, 14: 3, 15: 2,
   16: 2, 17: 2, 18: 3, 19: 3,
}

NOMBRES_RIESGOS = {
    1:  "Gestión estratégica",
    2:  "Planificación y seguimiento",
    3:  "Gestión de requisitos",
    4:  "Gestión de proyectos",
    5:  "Aseguramiento de calidad",
    6:  "Gestión de configuración",
    7:  "Medición y análisis",
    8:  "Gestión de proveedores",
    9:  "Acuerdos con proveedores",
    10: "Infraestructura",
    11: "Recursos humanos",
    12: "Gestión del conocimiento",
    13: "Comunicación organizacional",
    14: "Gestión del desempeño",
    15: "Mejora continua",
    16: "Gestión de cambios",
    17: "Análisis de decisiones",
    18: "Riesgos operacionales",
    19: "Gobierno corporativo",
}

_PROB_MIN, _PROB_MAX = 1, 3
_IMP_MIN,  _IMP_MAX  = 1, 3


# ══════════════════════════════════════════════════════════════════════════════
# SECCIÓN 5: CÁLCULOS DE RIESGOS
# ══════════════════════════════════════════════════════════════════════════════

def _calcular_promedio_ponderacion(df: pd.DataFrame, filas_excel: list) -> float:
    indices_0based = [f - 2 for f in filas_excel if 0 <= f - 2 < len(df)]
    if not indices_0based:
        return float("nan")
    valores = df.iloc[indices_0based]["PND_MADUREZ"]
    numericos = pd.to_numeric(valores, errors="coerce").dropna()
    return float("nan") if numericos.empty else round(numericos.mean(), 10)


def _calcular_promedio_pct(df: pd.DataFrame, filas_excel: list) -> float:
    indices_0based = [f - 2 for f in filas_excel if 0 <= f - 2 < len(df)]
    if not indices_0based:
        return 0.0
    niveles = df.iloc[indices_0based]["NVL_MADUREZ"]
    ponderaciones = niveles.map(PONDERACION).dropna()
    return 0.0 if ponderaciones.empty else round(float(ponderaciones.mean()) * 100, 2)


def _calcular_probabilidad(promedio_ponderacion: float) -> int:
    if math.isnan(promedio_ponderacion):
        return 3
    prom_entero = int(round(promedio_ponderacion))
    if prom_entero <= 1:
        return 3
    elif prom_entero == 2:
        return 2
    else:
        return 1


def _normalizar(valor: float, minimo: float, maximo: float) -> float:
    if maximo == minimo:
        return 0.0
    return round((valor - minimo) / (maximo - minimo) * 100, 2)


# ══════════════════════════════════════════════════════════════════════════════
# SECCIÓN 6: PIPELINE DE RIESGOS
# ══════════════════════════════════════════════════════════════════════════════

def build_tabla_riesgos(df: pd.DataFrame) -> pd.DataFrame:
    registros = []
    for rid, filas in REGLAS_RIESGOS.items():
        label = f"R{rid}"
        prom_pond   = _calcular_promedio_ponderacion(df, filas)
        prob        = _calcular_probabilidad(prom_pond)
        imp         = IMPACTOS_RIESGOS[rid]
        ri          = prob * imp
        prom_pct    = _calcular_promedio_pct(df, filas)
        x_pct = _normalizar(imp,  _IMP_MIN,  _IMP_MAX)
        y_pct = _normalizar(prob, _PROB_MIN, _PROB_MAX)
        registros.append({
            "N_RIESGO":         label,
            "RIESGO":           NOMBRES_RIESGOS[rid],
            "LINEAS":           ", ".join(str(f) for f in filas),
            "PROM_MADUREZ_PCT": prom_pct,
            "PROBABILIDAD":     prob,
            "IMPACTO":          imp,
            "RIESGO_INHERENTE": ri,
            "X_HEATMAP":        x_pct,
            "Y_HEATMAP":        y_pct,
        })
    return pd.DataFrame(registros)


def build_heatmap_payload(tabla: pd.DataFrame) -> dict:
    processes = []
    for _, row in tabla.iterrows():
        processes.append({
            "name":    f"{row['N_RIESGO']} · {row['RIESGO']}",
            "label":   row["N_RIESGO"],
            "avgPct":  float(row["PROM_MADUREZ_PCT"]),
            "xPct":    float(row["X_HEATMAP"]),
            "yPct":    float(row["Y_HEATMAP"]),
            "impacto": int(row["IMPACTO"]),
            "prob":    int(row["PROBABILIDAD"]),
            "ri":      int(row["RIESGO_INHERENTE"]),
        })
    global_avg = round(float(tabla["PROM_MADUREZ_PCT"].mean()), 2)
    return {"processes": processes, "globalAvg": global_avg}


# ══════════════════════════════════════════════════════════════════════════════
# SECCIÓN 7: MOTOR DE ANÁLISIS LLAMA — PROMPTS Y LLAMADAS
# ══════════════════════════════════════════════════════════════════════════════

def _call_llama(system_prompt: str, user_prompt: str,
                model: str = None,
                temperature: float = None,
                num_ctx: int = None) -> str:
    from langchain_core.messages import SystemMessage, HumanMessage
    _model  = model       if model       is not None else OLLAMA_MODEL
    _temp   = temperature if temperature is not None else OLLAMA_TEMP
    _ctx    = num_ctx     if num_ctx     is not None else OLLAMA_CTX
    llm = ChatOllama(model=_model, temperature=_temp, num_ctx=_ctx,
                     base_url=OLLAMA_BASE_URL)
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt),
    ]
    response = llm.invoke(messages)
    return response.content.strip()


def _extract_json(raw: str) -> dict:
    raw = re.sub(r'^```(?:json)?\s*', '', raw)
    raw = re.sub(r'\s*```$', '', raw)
    raw = raw.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass
    json_match = re.search(r'\{.*\}', raw, re.DOTALL)
    if json_match:
        return json.loads(json_match.group(0))
    raise ValueError(f"No se encontró JSON válido en: {raw[:200]}...")


# ── NIST CSF 2.0 / ISO 27001 reference map por función ────────────────────────
NIST_ISO_MAP = {
    "GOBERNAR":    {
        "nist": ["GV.OC-01", "GV.OC-02", "GV.RM-01", "GV.RM-02", "GV.SC-01"],
        "iso":  ["A.5.1", "A.5.2", "A.5.36", "A.6.1", "A.5.4"]
    },
    "IDENTIFICAR": {
        "nist": ["ID.AM-01", "ID.AM-02", "ID.AM-05", "ID.RA-01", "ID.RA-05"],
        "iso":  ["A.5.9", "A.5.10", "A.8.1", "A.8.2", "A.5.14"]
    },
    "PROTEGER":    {
        "nist": ["PR.AA-01", "PR.AA-03", "PR.DS-01", "PR.PS-01", "PR.IR-01"],
        "iso":  ["A.5.15", "A.5.17", "A.8.3", "A.8.5", "A.8.7"]
    },
    "DETECTAR":    {
        "nist": ["DE.AE-01", "DE.AE-02", "DE.CM-01", "DE.CM-09", "DE.AE-06"],
        "iso":  ["A.8.15", "A.8.16", "A.5.25", "A.5.26", "A.8.17"]
    },
    "RESPONDER":   {
        "nist": ["RS.MA-01", "RS.MA-02", "RS.CO-02", "RS.AN-03", "RS.MI-01"],
        "iso":  ["A.5.24", "A.5.25", "A.5.26", "A.6.8", "A.5.28"]
    },
    "RECUPERAR":   {
        "nist": ["RC.RP-01", "RC.RP-02", "RC.CO-03", "RC.CO-04", "RC.RP-05"],
        "iso":  ["A.5.29", "A.5.30", "A.8.13", "A.8.14", "A.5.30"]
    },
}

def _get_refs(process_name: str) -> dict:
    proc_upper = process_name.upper()
    for k, v in NIST_ISO_MAP.items():
        if k in proc_upper:
            return v
    return {
        "nist": ["GV.OC-01", "ID.AM-01", "PR.AA-01", "DE.AE-01", "RS.MA-01"],
        "iso":  ["A.5.1", "A.5.9", "A.8.1", "A.8.16", "A.5.24"]
    }


# ── PROMPT: Executive Summary ──────────────────────────────────────────────────
def _build_executive_summary_prompt(global_avg: float, risk_avg: float,
                                     processes: list, top_critical: list,
                                     top_risks: list) -> tuple:
    crit_ctx = "\n".join([
        f"- {p['name']}: {p['avgPct']:.1f}% (nivel dominante: {max(p['maturityCounts'], key=p['maturityCounts'].get)})"
        for p in top_critical
    ])
    risk_ctx = "\n".join([
        f"- {r['name']}: Riesgo Inherente={r['ri']} (Prob={r.get('prob','-')}, Impacto={r.get('impacto','-')})"
        for r in top_risks[:3]
    ])

    if global_avg < 30:
        estado = "CRÍTICO — controles inexistentes o ausentes en la mayoría de procesos"
        apertura = "La organización enfrenta una exposición cibernética severa que amenaza directamente la continuidad del negocio."
        directiva = "La junta debe aprobar recursos de emergencia y designar un responsable ejecutivo de remediación en los próximos 30 días."
    elif global_avg < 50:
        estado = "DEFICIENTE — prácticas informales sin estandarización ni control formal"
        apertura = "La postura de ciberseguridad actual no ofrece garantías suficientes frente a amenazas de mediana sofisticación."
        directiva = "Se requiere aprobación inmediata de un programa de mejora con presupuesto asignado y revisión mensual de avances."
    elif global_avg < 70:
        estado = "EN PROGRESO — bases establecidas pero con brechas críticas en procesos clave"
        apertura = "La organización ha iniciado su camino hacia la madurez en ciberseguridad, pero subsisten brechas que representan riesgos no aceptables."
        directiva = "La junta debe comprometer recursos adicionales para cerrar las brechas identificadas en un horizonte de 6 a 12 meses."
    else:
        estado = "AVANZADO — procesos formalizados con oportunidades de optimización"
        apertura = "La organización mantiene una postura de ciberseguridad sólida, con procesos maduros que requieren optimización continua."
        directiva = "El foco debe trasladarse hacia la automatización de controles, threat intelligence y preparación ante amenazas avanzadas persistentes."

    system = (
        "Eres un CISO con 15 años de experiencia certificado en CISSP, CISM e ISO 27001 Lead Auditor. "
        "Redactas informes ejecutivos para juntas directivas: lenguaje directo, orientado al impacto en el negocio. "
        "NUNCA usas frases genéricas. Cada frase aporta información accionable. "
        "REGLA ABSOLUTA: responde SOLO con el texto del resumen ejecutivo, sin JSON, sin títulos, sin listas."
    )

    user = f"""Escribe el resumen ejecutivo de un informe de ciberseguridad.

CONTEXTO:
- Estado global: {estado}
- Índice de madurez CMMI: {global_avg:.1f}% (marco NIST CSF 2.0 + ISO 27001)
- Índice de riesgos inherentes: {risk_avg:.1f}%
- Procesos evaluados: {len(processes)}

PROCESOS MÁS CRÍTICOS:
{crit_ctx}

RIESGOS DE MAYOR EXPOSICIÓN:
{risk_ctx}

ESTRUCTURA (prosa corrida, sin títulos ni bullets):
1. APERTURA (1-2 oraciones): Adapta esta base con datos reales: "{apertura}"
2. DIAGNÓSTICO (3-4 oraciones): qué revelan los datos, nombra procesos específicos, niveles de madurez.
3. RIESGOS PRIORITARIOS (2-3 oraciones): riesgos de mayor exposición e implicación operacional concreta.
4. DIRECTIVA EJECUTIVA (1-2 oraciones): adapta esta base: "{directiva}"

REQUISITOS:
- Entre 200 y 240 palabras
- Tono ejecutivo, directo
- Menciona al menos 2 procesos por nombre real
- Incluye al menos 1 referencia a un riesgo específico
- NO generes JSON"""

    return system, user


# ── PROMPT: Análisis profundo por proceso individual ──────────────────────────
def _build_process_analysis_prompt(process: dict, rank: int,
                                    heatmap_processes: list) -> tuple:
    worst_sub = min(process['subs'], key=lambda s: s['pctg'])
    best_sub  = max(process['subs'], key=lambda s: s['pctg'])
    dominant  = max(process['maturityCounts'], key=process['maturityCounts'].get)
    gap       = best_sub['pctg'] - worst_sub['pctg']

    related_risk = next(
        (r for r in heatmap_processes if process['name'].split()[0].upper() in r['name'].upper()),
        None
    )
    risk_ctx = ""
    if related_risk:
        risk_ctx = (
            f"\nRIESGO VINCULADO: {related_risk['name']} "
            f"(RI={related_risk['ri']}, Prob={related_risk.get('prob','-')}, "
            f"Imp={related_risk.get('impacto','-')}, madurez={related_risk['avgPct']:.1f}%)"
        )

    refs = _get_refs(process['name'])

    system = (
        "Eres un consultor sénior de ciberseguridad con 15+ años, especialista en NIST CSF 2.0 e ISO/IEC 27001:2022. "
        "Tu análisis es técnico, accionable y riguroso. Diriges el análisis a equipos de seguridad que deben ejecutar mejoras. "
        "REGLA ABSOLUTA: responde SOLO con el objeto JSON solicitado, sin texto previo ni posterior, "
        "sin bloques markdown. El JSON debe ser válido y parseable directamente."
    )

    user = f"""Genera el análisis experto completo del proceso de ciberseguridad #{rank}.

DATOS DEL PROCESO:
- Nombre: {process['name']}
- Cumplimiento promedio: {process['avgPct']:.1f}%
- Nivel dominante: {dominant}
- Subproceso MÁS CRÍTICO: {worst_sub['subproceso']} ({worst_sub['pctg']:.1f}%)
- Subproceso MEJOR: {best_sub['subproceso']} ({best_sub['pctg']:.1f}%)
- Brecha interna: {gap:.1f} puntos porcentuales
- Distribución madurez: {json.dumps(process['maturityCounts'], ensure_ascii=False)}{risk_ctx}

REFERENCIAS NORMATIVAS (usa exactamente estas):
- NIST CSF 2.0: {refs['nist']}
- ISO/IEC 27001:2022: {refs['iso']}

GENERA este JSON exacto (sin texto adicional, sin ```json):
{{
  "process": "{process['name']}",
  "why_critical": "2-3 frases que expliquen por qué este proceso al {process['avgPct']:.1f}% representa riesgo inherente alto. Menciona el subproceso más débil ({worst_sub['subproceso']}), el nivel dominante ({dominant}) y el impacto concreto en la postura de seguridad.",
  "normative_references": {{
    "nist_csf": {json.dumps(refs['nist'][:4])},
    "iso_27001": {json.dumps(refs['iso'][:4])}
  }},
  "recommended_actions": [
    "Acción 1 concreta y medible enfocada en {worst_sub['subproceso']} alineada a {refs['nist'][0]}",
    "Acción 2: implementar control específico de {refs['iso'][0]} para formalizar el proceso",
    "Acción 3: establecer métricas de seguimiento mensual para {worst_sub['subproceso']}",
    "Acción 4: documentar política formal que eleve el nivel dominante de {dominant} al siguiente nivel",
    "Acción 5: designar responsable nominado con KPIs medibles en 90 días"
  ],
  "how_to_implement": "Fase 1 (semanas 1-4): diagnóstico detallado y quick wins en {worst_sub['subproceso']}. Fase 2 (semanas 5-10): implementación de controles formales documentados según {refs['nist'][1]}. Fase 3 (semanas 11-16): validación, métricas y presentación a dirección. Métrica de éxito: cumplimiento ≥60% al cierre de Fase 3.",
  "success_metrics": [
    "Técnica: {worst_sub['subproceso']} alcanza ≥60% de cumplimiento en 90 días",
    "Negocio: reducción del riesgo inherente asociado en al menos 2 puntos en la próxima evaluación trimestral",
    "Operacional: 100% de controles documentados y asignados a responsable antes de semana 8"
  ],
  "dashboard_insight": "1-2 frases de análisis del radar chart de este proceso: qué patrón de madurez muestra ({dominant} dominante), qué indica la brecha de {gap:.1f} puntos entre mejor y peor subproceso, y qué debe observar el equipo técnico en el gráfico.",
  "maturity_distribution_insight": "1-2 frases que interpreten la distribución de niveles de madurez {json.dumps(process['maturityCounts'], ensure_ascii=False)} bajo la lente NIST CSF: qué concentración de preguntas en cada nivel revela sobre la madurez del proceso y qué implica para la hoja de ruta."
}}"""

    return system, user


# ── PROMPT: Análisis del Dashboard General (radar + barras apiladas) ──────────
def _build_dashboard_analysis_prompt(global_avg: float, processes: list,
                                      risk_avg: float) -> tuple:
    proc_summary = "\n".join([
        f"- {p['name']}: {p['avgPct']:.1f}% (dominante: {max(p['maturityCounts'], key=p['maturityCounts'].get)})"
        for p in processes
    ])

    # Calcular distribución global de madurez
    total_counts = {"1-Inexistente": 0, "2-Inicial": 0, "3-Definida": 0, "4-Optimizada": 0}
    for p in processes:
        for lvl, cnt in p['maturityCounts'].items():
            if lvl in total_counts:
                total_counts[lvl] += cnt

    system = (
        "Eres un consultor sénior de ciberseguridad especialista en NIST CSF 2.0 e ISO/IEC 27001:2022. "
        "Analizas dashboards de madurez y produces conclusiones técnicas accionables. "
        "REGLA ABSOLUTA: responde SOLO con el objeto JSON, sin texto adicional, sin markdown."
    )

    user = f"""Analiza el dashboard general de madurez en ciberseguridad.

DATOS GLOBALES:
- Índice global CMMI: {global_avg:.1f}%
- Índice global de riesgos: {risk_avg:.1f}%
- Distribución global de madurez: {json.dumps(total_counts, ensure_ascii=False)}

PROCESOS (orden NIST CSF):
{proc_summary}

GENERA este JSON exacto (sin texto adicional):
{{
  "radar_insight": "2-3 frases que interpreten el gráfico radar de cumplimiento por proceso. Identifica el patrón de fortalezas y debilidades bajo NIST CSF 2.0. Menciona qué funciones del framework están más rezagadas y el impacto en la postura de seguridad global.",
  "stacked_bar_insight": "2-3 frases que analicen la distribución de niveles de madurez por proceso. Qué dice el predominio de niveles 1-Inexistente o 2-Inicial sobre la capacidad de la organización para gestionar incidentes. Referencia ISO 27001 A.5.1 o controles relevantes.",
  "kpi_insight": "1-2 frases sobre los KPIs de cumplimiento por proceso: cuál es el proceso ancla de estabilidad y cuáles son los que más arrastran el índice global hacia abajo.",
  "global_posture": "2-3 frases de diagnóstico general de la postura de ciberseguridad bajo el doble lente NIST CSF 2.0 + ISO/IEC 27001:2022. Incluye una referencia a qué tan preparada está la organización para responder a un incidente real con este nivel de madurez.",
  "priority_actions": [
    "Acción prioritaria 1 basada en el proceso con menor cumplimiento, con referencia NIST CSF específica",
    "Acción prioritaria 2 para mejorar la distribución de madurez, con referencia ISO 27001 específica",
    "Acción prioritaria 3 de carácter estratégico para elevar el índice global en 12 meses"
  ]
}}"""

    return system, user


# ── PROMPT: Análisis del Heatmap de Riesgos ───────────────────────────────────
def _build_heatmap_analysis_prompt(heatmap_data: dict) -> tuple:
    risks = heatmap_data['processes']
    global_avg = heatmap_data['globalAvg']

    # Top 5 riesgos por RI
    top_risks = sorted(risks, key=lambda x: x['ri'], reverse=True)[:5]
    top_ctx = "\n".join([
        f"- {r['name']}: RI={r['ri']} (Prob={r.get('prob','-')}, Imp={r.get('impacto','-')}, madurez={r['avgPct']:.1f}%)"
        for r in top_risks
    ])

    # Riesgos en zona crítica (RI >= 6)
    critical = [r for r in risks if r['ri'] >= 6]
    high = [r for r in risks if 4 <= r['ri'] < 6]
    medium = [r for r in risks if r['ri'] < 4]

    system = (
        "Eres un consultor sénior de gestión de riesgos de ciberseguridad, especialista en NIST CSF 2.0 e ISO/IEC 27001:2022 "
        "con experiencia en análisis de heatmaps de riesgo para juntas directivas. "
        "REGLA ABSOLUTA: responde SOLO con el objeto JSON, sin texto adicional, sin markdown."
    )

    user = f"""Analiza el heatmap de riesgos de ciberseguridad.

DATOS DEL HEATMAP:
- Total de riesgos evaluados: {len(risks)}
- Índice global de madurez de riesgos: {global_avg:.1f}%
- Riesgos en zona CRÍTICA (RI ≥ 6): {len(critical)} — {[r['name'] for r in critical]}
- Riesgos en zona ALTA (RI 4-5): {len(high)} — {[r['name'] for r in high]}
- Riesgos en zona MEDIA/BAJA (RI < 4): {len(medium)}

TOP 5 RIESGOS POR RIESGO INHERENTE:
{top_ctx}

GENERA este JSON exacto (sin texto adicional):
{{
  "heatmap_reading": "2-3 frases que expliquen cómo leer el heatmap: qué significa la posición de los puntos (eje X = Impacto, eje Y = Probabilidad), qué zonas son de atención inmediata y cómo se correlaciona el color del punto con el nivel de madurez del control.",
  "critical_zone_analysis": "2-3 frases de análisis de los riesgos en zona crítica (RI ≥ 6). Nombra los riesgos más expuestos y explica el impacto operacional concreto si alguno de estos riesgos se materializa. Referencia NIST CSF GV.RM-01 o ISO 27001 A.5.36.",
  "risk_clusters": "1-2 frases identificando si existen clusters o patrones en el heatmap: ¿están los riesgos concentrados en alta probabilidad, alto impacto, o distribuidos? ¿Qué dice este patrón sobre los controles actuales?",
  "remediation_priority": "2-3 frases con la estrategia de priorización de remediación: qué riesgos atacar primero (mayor RI + menor madurez), cuáles pueden gestionarse en segunda fase, y cuáles son aceptables con monitoreo.",
  "executive_risk_message": "1-2 frases en lenguaje ejecutivo (para la junta) que resuman el nivel de exposición al riesgo de la organización y la urgencia de acción sin usar jerga técnica.",
  "iso_nist_controls": [
    "Control NIST/ISO específico para el riesgo de mayor RI con descripción de acción concreta",
    "Control NIST/ISO específico para el segundo riesgo de mayor RI con descripción de acción concreta",
    "Control NIST/ISO específico para reducir la probabilidad en el cluster de mayor concentración"
  ]
}}"""

    return system, user


# ── PROMPT: Informe General de Conclusiones (slide final) ────────────────────
def _build_conclusions_prompt(global_avg: float, risk_avg: float,
                               processes: list, heatmap_data: dict) -> tuple:
    risks = heatmap_data['processes']
    top_risks = sorted(risks, key=lambda x: x['ri'], reverse=True)[:3]
    worst_procs = processes[:3]
    best_procs  = processes[-2:]

    worst_ctx = "\n".join([f"- {p['name']}: {p['avgPct']:.1f}%" for p in worst_procs])
    best_ctx  = "\n".join([f"- {p['name']}: {p['avgPct']:.1f}%" for p in best_procs])
    risk_ctx  = "\n".join([
        f"- {r['name']}: RI={r['ri']} (madurez {r['avgPct']:.1f}%)"
        for r in top_risks
    ])

    # Nivel global
    if global_avg < 30:
        nivel = "CRÍTICO"
        horizonte = "inmediato (0-3 meses)"
    elif global_avg < 50:
        nivel = "DEFICIENTE"
        horizonte = "urgente (3-6 meses)"
    elif global_avg < 70:
        nivel = "EN PROGRESO"
        horizonte = "prioritario (6-12 meses)"
    else:
        nivel = "AVANZADO"
        horizonte = "de optimización (12-18 meses)"

    system = (
        "Eres un CISO sénior con 15+ años de experiencia, certificado en CISSP, CISM e ISO 27001 Lead Auditor. "
        "Produces informes de conclusiones que integran visión ejecutiva y técnica. "
        "REGLA ABSOLUTA: responde SOLO con el objeto JSON, sin texto adicional, sin markdown."
    )

    user = f"""Genera el informe de conclusiones y cierre de la evaluación de madurez en ciberseguridad.

RESUMEN DE LA EVALUACIÓN:
- Estado global: {nivel}
- Índice global CMMI: {global_avg:.1f}% bajo NIST CSF 2.0 + ISO/IEC 27001:2022
- Índice global de riesgos: {risk_avg:.1f}%
- Horizonte de remediación recomendado: {horizonte}

PROCESOS MÁS CRÍTICOS:
{worst_ctx}

PROCESOS MÁS MADUROS:
{best_ctx}

RIESGOS DE MAYOR EXPOSICIÓN:
{risk_ctx}

GENERA este JSON exacto (sin texto adicional):
{{
  "overall_diagnosis": "3-4 frases de diagnóstico integral de la organización bajo NIST CSF 2.0 e ISO/IEC 27001:2022. Qué posición ocupa la organización en el espectro de madurez, qué brechas estructurales existen y cuál es la implicación para la continuidad del negocio.",
  "key_findings": [
    "Hallazgo 1: fortaleza identificada con nombre de proceso y dato específico",
    "Hallazgo 2: brecha crítica más relevante con nombre de proceso y dato específico",
    "Hallazgo 3: patrón de riesgo transversal identificado en múltiples procesos",
    "Hallazgo 4: riesgo inherente más expuesto y su implicación operacional"
  ],
  "strategic_recommendations": [
    {{
      "titulo": "Recomendación estratégica 1 (título corto)",
      "descripcion": "2-3 frases: qué hacer, cómo, timeline estimado y KPI de éxito. Referencia NIST o ISO específica.",
      "prioridad": "ALTA",
      "timeline": "0-90 días"
    }},
    {{
      "titulo": "Recomendación estratégica 2 (título corto)",
      "descripcion": "2-3 frases: qué hacer, cómo, timeline estimado y KPI de éxito. Referencia NIST o ISO específica.",
      "prioridad": "ALTA",
      "timeline": "90-180 días"
    }},
    {{
      "titulo": "Recomendación estratégica 3 (título corto)",
      "descripcion": "2-3 frases: qué hacer, cómo, timeline estimado y KPI de éxito. Referencia NIST o ISO específica.",
      "prioridad": "MEDIA",
      "timeline": "180-365 días"
    }}
  ],
  "roadmap_summary": "2-3 frases que describan el roadmap de mejora: fases principales, hitos clave y objetivo de índice global al cabo de 12 meses. Menciona la meta de cumplimiento esperada y el mecanismo de seguimiento recomendado.",
  "closing_message": "1-2 frases de cierre en tono ejecutivo que transmitan la urgencia o la oportunidad según el nivel {nivel}, sin caer en clichés. Debe sonar a CISO dirigiéndose a la junta directiva.",
  "compliance_gap_summary": {{
    "nist_csf_gap": "1-2 frases sobre las principales brechas frente al NIST CSF 2.0: qué funciones del framework están más rezagadas.",
    "iso_27001_gap": "1-2 frases sobre las principales brechas frente a ISO/IEC 27001:2022: qué dominios de control requieren atención inmediata."
  }}
}}"""

    return system, user


# ══════════════════════════════════════════════════════════════════════════════
# SECCIÓN 8: FALLBACKS
# ══════════════════════════════════════════════════════════════════════════════

def _fallback_executive_summary(global_avg: float) -> str:
    nivel = "crítico" if global_avg < 50 else "en desarrollo" if global_avg < 75 else "avanzado"
    return (
        f"La evaluación de madurez en ciberseguridad revela una postura {nivel} "
        f"con un índice global del {global_avg:.1f}% bajo el marco NIST CSF 2.0 e ISO/IEC 27001:2022. "
        "Existen brechas materiales en los procesos de gobierno, identificación y protección "
        "que exponen a la organización a riesgos operacionales y regulatorios significativos. "
        "La junta directiva debe aprobar un plan de remediación con recursos asignados y "
        "revisión trimestral de avances. Se recomienda iniciar de inmediato con los tres "
        "procesos de menor cumplimiento, estableciendo métricas de éxito claras y responsables "
        "nominados para cada iniciativa."
    )


def _fallback_process_analysis(proc: dict) -> dict:
    worst = min(proc['subs'], key=lambda s: s['pctg'])
    refs = _get_refs(proc['name'])
    return {
        "process": proc['name'],
        "why_critical": (
            f"{proc['name']} opera al {proc['avgPct']:.1f}% de cumplimiento, "
            f"con '{worst['subproceso']}' como subproceso más débil ({worst['pctg']:.1f}%). "
            "La ausencia de controles formales en este dominio aumenta la exposición a incidentes no detectados."
        ),
        "normative_references": {
            "nist_csf": refs['nist'][:4],
            "iso_27001": refs['iso'][:4]
        },
        "recommended_actions": [
            f"Documentar el estado actual de '{worst['subproceso']}' y definir controles mínimos requeridos",
            "Asignar responsable nominado para el proceso con objetivos medibles en 90 días",
            "Establecer revisión mensual de cumplimiento con reporte a dirección",
            "Implementar al menos un control preventivo formal antes del cierre del trimestre",
            f"Alinear los controles al {refs['nist'][0]} del NIST CSF 2.0"
        ],
        "how_to_implement": "Fase 1 (semanas 1-4): inventario y brecha; Fase 2 (semanas 5-10): controles básicos; Fase 3 (semanas 11-16): validación y métricas.",
        "success_metrics": [
            f"Técnica: {worst['subproceso']} alcanza ≥55% en 90 días",
            "Negocio: proceso incluido en siguiente ciclo de auditoría interna con hallazgos cerrados",
            "Operacional: controles documentados y asignados antes de semana 8"
        ],
        "dashboard_insight": f"El proceso {proc['name']} muestra un patrón de madurez desigual que evidencia capacidad instalada parcial pero sin estandarización formal.",
        "maturity_distribution_insight": "La distribución de niveles de madurez indica que los controles existentes no están formalizados ni son repetibles bajo estándares NIST CSF 2.0."
    }


def _fallback_dashboard_analysis(global_avg: float, processes: list) -> dict:
    worst = processes[0]
    best  = processes[-1]
    return {
        "radar_insight": f"El radar de cumplimiento revela una postura asimétrica con {best['name']} como proceso ancla ({best['avgPct']:.1f}%) y {worst['name']} como el dominio más crítico ({worst['avgPct']:.1f}%), evidenciando brechas estructurales bajo NIST CSF 2.0.",
        "stacked_bar_insight": "La distribución de niveles de madurez muestra concentración en los niveles 1-Inexistente y 2-Inicial, indicando que la organización aún no cuenta con controles formalizados y repetibles en la mayoría de sus procesos.",
        "kpi_insight": f"{best['name']} actúa como proceso ancla de estabilidad, mientras que {worst['name']} arrastra el índice global hacia abajo con su {worst['avgPct']:.1f}% de cumplimiento.",
        "global_posture": f"La postura global al {global_avg:.1f}% bajo NIST CSF 2.0 e ISO/IEC 27001:2022 indica que la organización no cuenta con capacidad suficiente para detectar y responder a un incidente de seguridad de mediana sofisticación.",
        "priority_actions": [
            f"Remediar {worst['name']} con plan formal de 90 días alineado a NIST CSF GV.OC-01",
            "Elevar el nivel 1-Inexistente a 2-Inicial en todos los procesos antes del próximo trimestre, alineado a ISO 27001 A.5.1",
            f"Establecer un programa de mejora continua para elevar el índice global del {global_avg:.1f}% al 60% en 12 meses"
        ]
    }


def _fallback_heatmap_analysis(heatmap_data: dict) -> dict:
    risks = heatmap_data['processes']
    critical = [r for r in risks if r['ri'] >= 6]
    top = sorted(risks, key=lambda x: x['ri'], reverse=True)[0]
    return {
        "heatmap_reading": "El heatmap posiciona cada riesgo según su probabilidad de ocurrencia (eje Y) e impacto potencial (eje X). Los puntos en la esquina superior derecha representan la zona de mayor exposición y requieren remediación inmediata.",
        "critical_zone_analysis": f"Se identifican {len(critical)} riesgos en zona crítica (RI ≥ 6). El riesgo '{top['name']}' con RI={top['ri']} representa la mayor exposición y su materialización podría comprometer la continuidad operacional de la organización.",
        "risk_clusters": "Los riesgos presentan concentración en la zona de alta probabilidad, indicando que los controles preventivos son insuficientes o inexistentes en múltiples dominios.",
        "remediation_priority": "La estrategia de remediación debe priorizar los riesgos con RI ≥ 6 y menor madurez en los próximos 90 días, seguidos de los riesgos con RI 4-5 en el período de 90-180 días.",
        "executive_risk_message": f"La organización tiene {len(critical)} áreas de riesgo crítico que requieren acción inmediata para proteger sus activos clave y garantizar la continuidad del negocio.",
        "iso_nist_controls": [
            f"NIST GV.RM-01 + ISO A.5.36: implementar marco formal de gestión de riesgos para '{top['name']}'",
            "NIST ID.RA-01 + ISO A.8.8: realizar evaluación de vulnerabilidades técnicas en los sistemas de mayor exposición",
            "NIST PR.AA-01 + ISO A.5.15: fortalecer controles de acceso para reducir la probabilidad en el cluster de mayor concentración"
        ]
    }


def _fallback_conclusions(global_avg: float, risk_avg: float, processes: list) -> dict:
    worst = processes[0]
    best  = processes[-1]
    nivel = "CRÍTICO" if global_avg < 30 else "DEFICIENTE" if global_avg < 50 else "EN PROGRESO" if global_avg < 70 else "AVANZADO"
    return {
        "overall_diagnosis": f"La organización alcanza un índice global de madurez del {global_avg:.1f}% bajo NIST CSF 2.0 e ISO/IEC 27001:2022, situándose en estado {nivel}. Existen brechas estructurales que limitan la capacidad de detección y respuesta ante incidentes. La continuidad del negocio está expuesta en los dominios de menor cumplimiento.",
        "key_findings": [
            f"Fortaleza: {best['name']} alcanza {best['avgPct']:.1f}%, demostrando capacidad instalada y base para replicar prácticas a otros procesos",
            f"Brecha crítica: {worst['name']} con {worst['avgPct']:.1f}% representa el talón de Aquiles de la postura de seguridad",
            "Patrón transversal: predominio de niveles 1-Inexistente y 2-Inicial en múltiples procesos indica ausencia de estandarización",
            f"Riesgo sistémico: el índice de riesgos inherentes del {risk_avg:.1f}% expone a la organización a eventos de impacto alto sin controles mitigadores formales"
        ],
        "strategic_recommendations": [
            {
                "titulo": "Programa de remediación prioritaria",
                "descripcion": f"Implementar plan formal de remediación para {worst['name']} con sponsor ejecutivo, presupuesto asignado y revisión mensual. Alineado a NIST CSF GV.OC-01 e ISO 27001 A.5.1. KPI: cumplimiento ≥55% en 90 días.",
                "prioridad": "ALTA",
                "timeline": "0-90 días"
            },
            {
                "titulo": "Marco de gestión de riesgos continuo",
                "descripcion": "Establecer proceso formal de evaluación de riesgos trimestral con umbrales de aceptación aprobados por la junta. Alineado a NIST GV.RM-01 e ISO 27001 A.5.36. KPI: 100% de riesgos con plan de tratamiento documentado.",
                "prioridad": "ALTA",
                "timeline": "90-180 días"
            },
            {
                "titulo": "Roadmap de madurez a 12 meses",
                "descripcion": f"Definir hoja de ruta para elevar el índice global del {global_avg:.1f}% al 60% en 12 meses con hitos trimestrales medibles. Alineado a NIST CSF ID.IM-01 e ISO 27001 A.5.35. KPI: reporte trimestral ante comité de riesgos.",
                "prioridad": "MEDIA",
                "timeline": "180-365 días"
            }
        ],
        "roadmap_summary": f"El roadmap de mejora se estructura en tres fases: Fase 1 (0-90 días) remediación de brechas críticas, Fase 2 (90-180 días) estandarización de controles, Fase 3 (180-365 días) optimización y medición continua. Meta: índice global ≥60% al cabo de 12 meses con seguimiento mensual ante el comité de riesgos.",
        "closing_message": f"El nivel {nivel} de madurez actual no es aceptable para una organización que opera en entornos de amenaza creciente; la decisión de actuar hoy determinará la resiliencia del negocio mañana.",
        "compliance_gap_summary": {
            "nist_csf_gap": "Las funciones de Gobernar e Identificar presentan las mayores brechas bajo NIST CSF 2.0, indicando ausencia de gobierno formal y visibilidad insuficiente sobre los activos críticos.",
            "iso_27001_gap": "Los dominios de control de activos (A.5.9-A.5.14) y gestión de accesos (A.5.15-A.5.18) requieren atención inmediata bajo ISO/IEC 27001:2022."
        }
    }


# ══════════════════════════════════════════════════════════════════════════════
# SECCIÓN 9: ORQUESTADOR PRINCIPAL DE ANÁLISIS LLAMA
# ══════════════════════════════════════════════════════════════════════════════

def analyze_with_llama_advanced(cmmi_data: dict, heatmap_data: dict) -> dict:
    """
    Análisis experto completo con Llama usando llamadas separadas por sección:
    1. executive_summary          → resumen para slide ejecutiva
    2. critical_analysis          → análisis profundo por cada proceso (una llamada por proceso)
    3. dashboard_analysis         → interpretación del dashboard general
    4. heatmap_analysis           → interpretación del heatmap de riesgos
    5. conclusions                → informe general de conclusiones (slide final)
    6. global_recommendations     → derivadas de forma determinista
    """
    print("🧠 Solicitando análisis experto a Llama (modo completo)...")

    global_avg   = cmmi_data['globalAvg']
    processes    = cmmi_data['processes']
    top_critical = processes[:3]
    risks        = heatmap_data['processes']
    top_risks    = sorted(risks, key=lambda x: x['ri'], reverse=True)[:5]
    risk_avg     = heatmap_data['globalAvg']

    result = {
        "executive_summary":       "",
        "critical_analysis":       [],
        "dashboard_analysis":      {},
        "heatmap_analysis":        {},
        "conclusions":             {},
        "global_recommendations":  []
    }

    # ── 1. Executive Summary ─────────────────────────────────────────────────
    print("   📝 [1/5] Generando executive summary...")
    try:
        sys_exec, usr_exec = _build_executive_summary_prompt(
            global_avg, risk_avg, processes, top_critical, top_risks
        )
        raw_exec = _call_llama(sys_exec, usr_exec, temperature=0.45, num_ctx=3072)
        if raw_exec and len(raw_exec.split()) >= 80:
            result["executive_summary"] = raw_exec
        else:
            print(f"   ⚠️  Summary muy corto ({len(raw_exec.split())} palabras), usando fallback")
            result["executive_summary"] = _fallback_executive_summary(global_avg)
    except Exception as e:
        print(f"   ❌ Error en executive_summary: {e}")
        result["executive_summary"] = _fallback_executive_summary(global_avg)

    # ── 2. Análisis por proceso crítico (una llamada por proceso) ────────────
    print("   🔍 [2/5] Analizando procesos críticos...")
    for i, proc in enumerate(top_critical):
        print(f"      → Proceso {i+1}/3: {proc['name']}")
        try:
            sys_proc, usr_proc = _build_process_analysis_prompt(proc, i + 1, risks)
            raw_proc = _call_llama(sys_proc, usr_proc, temperature=0.2, num_ctx=3072)
            proc_analysis = _extract_json(raw_proc)
            # Garantizar campos obligatorios
            for field, default in [
                ("process", proc['name']),
                ("why_critical", f"{proc['name']} presenta {proc['avgPct']:.1f}% de cumplimiento."),
                ("normative_references", {"nist_csf": [], "iso_27001": []}),
                ("recommended_actions", []),
                ("how_to_implement", "Fase 1 (semanas 1-4): diagnóstico; Fase 2 (semanas 5-12): implementación."),
                ("success_metrics", []),
                ("dashboard_insight", ""),
                ("maturity_distribution_insight", ""),
            ]:
                proc_analysis.setdefault(field, default)
            result["critical_analysis"].append(proc_analysis)
        except Exception as e:
            print(f"      ❌ Error analizando {proc['name']}: {e}")
            result["critical_analysis"].append(_fallback_process_analysis(proc))

    # ── 3. Dashboard Analysis ────────────────────────────────────────────────
    print("   📊 [3/5] Analizando dashboard general...")
    try:
        sys_dash, usr_dash = _build_dashboard_analysis_prompt(global_avg, processes, risk_avg)
        raw_dash = _call_llama(sys_dash, usr_dash, temperature=0.3, num_ctx=3072)
        result["dashboard_analysis"] = _extract_json(raw_dash)
    except Exception as e:
        print(f"   ❌ Error en dashboard_analysis: {e}")
        result["dashboard_analysis"] = _fallback_dashboard_analysis(global_avg, processes)

    # ── 4. Heatmap Analysis ──────────────────────────────────────────────────
    print("   🔥 [4/5] Analizando heatmap de riesgos...")
    try:
        sys_heat, usr_heat = _build_heatmap_analysis_prompt(heatmap_data)
        raw_heat = _call_llama(sys_heat, usr_heat, temperature=0.3, num_ctx=3072)
        result["heatmap_analysis"] = _extract_json(raw_heat)
    except Exception as e:
        print(f"   ❌ Error en heatmap_analysis: {e}")
        result["heatmap_analysis"] = _fallback_heatmap_analysis(heatmap_data)

    # ── 5. Conclusions (slide final de informe general) ──────────────────────
    print("   📋 [5/5] Generando informe de conclusiones...")
    try:
        sys_conc, usr_conc = _build_conclusions_prompt(
            global_avg, risk_avg, processes, heatmap_data
        )
        raw_conc = _call_llama(sys_conc, usr_conc, temperature=0.35, num_ctx=4096)
        result["conclusions"] = _extract_json(raw_conc)
    except Exception as e:
        print(f"   ❌ Error en conclusions: {e}")
        result["conclusions"] = _fallback_conclusions(global_avg, risk_avg, processes)

    # ── 6. Global Recommendations (deterministas) ────────────────────────────
    print("   🌐 Construyendo recomendaciones globales...")
    worst_proc  = processes[0]
    second_proc = processes[1] if len(processes) > 1 else processes[0]
    top_risk    = top_risks[0] if top_risks else None

    result["global_recommendations"] = [
        (
            f"Priorizar la remediación de '{worst_proc['name']}' ({worst_proc['avgPct']:.1f}%) "
            f"y '{second_proc['name']}' ({second_proc['avgPct']:.1f}%) mediante un programa formal "
            "con sponsor ejecutivo, presupuesto asignado y revisión mensual ante la dirección."
        ),
        (
            f"Implementar un marco de gestión de riesgos continuo que incorpore los {len(risks)} riesgos "
            "evaluados, con foco inmediato en los de mayor riesgo inherente"
            + (f" ('{top_risk['name']}', RI={top_risk['ri']})" if top_risk else "")
            + ", estableciendo umbrales de tolerancia aprobados por la junta."
        ),
        (
            f"Elevar el índice global de madurez del {global_avg:.1f}% actual a un mínimo del 60% "
            "en los próximos 12 meses mediante un roadmap trimestral con hitos verificables, "
            "asignación de KPIs por proceso y reporte periódico al comité de riesgos."
        )
    ]

    print(f"   ✅ Análisis completo: summary={len(result['executive_summary'].split())} palabras, "
          f"{len(result['critical_analysis'])} procesos, dashboard + heatmap + conclusiones generadas")

    return result


# ══════════════════════════════════════════════════════════════════════════════
# SECCIÓN 10: GENERADOR PRINCIPAL
# ══════════════════════════════════════════════════════════════════════════════

def generate(csv_path: str, out_path: str):
    print(f"📂 Leyendo CSV: {csv_path}")
    df = load_and_clean(csv_path)
    print(f"   → {len(df)} filas cargadas, {df['PROC'].nunique()} procesos")

    data = build_report_data(df)
    print(f"📊 Índice global CMMI: {data['globalAvg']}%")

    tabla_riesgos = build_tabla_riesgos(df)
    heatmap_data  = build_heatmap_payload(tabla_riesgos)
    print(f"🔥 Índice global riesgos: {heatmap_data['globalAvg']}%")
    print(f"   → {len(heatmap_data['processes'])} riesgos procesados")

    print("🧠 Solicitando análisis experto a Llama...")
    analysis = analyze_with_llama_advanced(data, heatmap_data)
    print("   ✅ Análisis recibido")

    payload = {
        "data":        data,
        "heatmapData": heatmap_data,
        "analysis":    analysis,
        "outPath":     out_path,
        "logoB64":     _b64("Logo_blanco.png"),
        "iconB64":     _b64("favicon.png"),
    }

    result = subprocess.run(
        ["node", "2.POWER_POINT.js", json.dumps(payload)],
        cwd=os.getcwd(),
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        print("❌ Error en Node.js:")
        print(result.stderr)
        return

    print(f"✅ PPTX generado: {out_path}")


# ══════════════════════════════════════════════════════════════════════════════
# SECCIÓN 11: PUNTO DE ENTRADA
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="RBKT CMMI — Generador de informe PPTX desde CSV"
    )
    parser.add_argument("--csv", default=DEFAULT_CSV,
                        help=f"Ruta al CSV de evaluación (default: {DEFAULT_CSV})")
    parser.add_argument("--out", default=DEFAULT_OUT,
                        help=f"Ruta de salida del PPTX (default: {DEFAULT_OUT})")
    args, unknown = parser.parse_known_args()
    generate(args.csv, args.out)

📂 Leyendo CSV: 3.PRUEBA.csv
   → 66 filas cargadas, 5 procesos
📊 Índice global CMMI: 35.09%
🔥 Índice global riesgos: 40.96%
   → 19 riesgos procesados
🧠 Solicitando análisis experto a Llama...
🧠 Solicitando análisis experto a Llama (modo completo)...
   📝 [1/5] Generando executive summary...
   🔍 [2/5] Analizando procesos críticos...
      → Proceso 1/3: RESPONDER
      → Proceso 2/3: DETECTAR
